# MM26 Data Pipeline Explorer

This notebook is a guided tour of the entire data pipeline from raw source files through model-ready artifacts. Each section explains what the data is, where it comes from, and what transformations were applied.

---

## Pipeline Overview

```
Kaggle CSVs  --.
              +--> Bronze (raw parquet snapshots)
CBBD API    --'        |
                      v
                 Silver (cleaned, joined, one row per team-game)
                      |-- ELO ratings (MOV + carry-over + home court)
                      |-- Heat scores (actual vs expected margin)
                      '-- Consensus betting lines
                      |
                      v
                 Gold (up to 28 pairwise features for ML)
                      |-- XGBoost classifier (isotonically calibrated)
                      '-- Dynamic probability clipping (seed-aware)
                      |
                      v
                 submission.csv
```

| Layer | Files | Description |
|---|---|---|
| **Bronze/Kaggle** | `bronze/kaggle/*.parquet` | Kaggle CSVs frozen as parquet snapshots |
| **Bronze/CBBD** | `bronze/cbbd/*.parquet` | API data: games, box scores, lines |
| **Silver** | `silver/*.parquet` | Cleaned and joined canonical team-game tables |
| **Gold** | `gold/*.parquet` | Aggregated team-season and pairwise features |

All files are read from `artifacts/latest/`, which points to the most recent pipeline run.

In [3]:
from __future__ import annotations
import json
from pathlib import Path

import polars as pl

ROOT = Path("..").resolve()
BASE = ROOT / "artifacts" / "latest"

required_paths = [
    BASE / "run_manifest.json",
    BASE / "reports" / "model_performance_summary.json",
    BASE / "reports" / "gold_quality_summary.json",
    BASE / "submission.csv",
    BASE / "gold" / "pairwise_features.parquet",
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    print("Missing required artifacts:")
    for p in missing:
        print(f"  - {p}")
    raise FileNotFoundError("One or more required artifacts are missing")

manifest = json.loads((BASE / "run_manifest.json").read_text(encoding="utf-8"))
model_summary = json.loads((BASE / "reports" / "model_performance_summary.json").read_text(encoding="utf-8"))
gold_summary = json.loads((BASE / "reports" / "gold_quality_summary.json").read_text(encoding="utf-8"))

print(f"Run ID        : {manifest['run_id']}")
print(f"Mode          : {manifest['mode']}")
print(f"Target season : {manifest['target_season']}")
print(f"CBBD seasons  : {manifest['ingest']['cbbd']['historical_seasons']}")
print(f"M holdout Brier: {model_summary['model_stats']['M']['holdout_brier']:.5f}")
print()
print("CBBD dataset row counts:")
for name, info in manifest["ingest"]["cbbd"]["datasets"].items():
    print(f"  {name:15s}  status={info['status']:7s}  rows={info['rows']:,}")

Run ID        : 20260319T040334Z_fbfc3ea8
Mode          : model_only
Target season : 2026
CBBD seasons  : [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
M holdout Brier: 0.13221

CBBD dataset row counts:
  games            status=error    rows=0
  game_teams       status=error    rows=0
  lines            status=error    rows=0


## Pipeline Health Dashboard

This quick dashboard summarizes whether the latest run is healthy before drilling into layer-by-layer details.

It checks:
- mapping coverage from `team_id_map_summary.json`
- feature completeness in `pairwise_features.parquet`
- prediction contract in `submission.csv`
- pass rates from `gold_quality_summary.json`

In [4]:
team_id_map_summary = json.loads((BASE / "reports" / "team_id_map_summary.json").read_text(encoding="utf-8"))
pairwise = pl.read_parquet(BASE / "gold" / "pairwise_features.parquet")
submission = pl.read_csv(BASE / "submission.csv")

feature_cols = [
    c for c in pairwise.columns
    if c not in {"ID", "season", "team_low", "team_high", "sex", "label", "target_low_wins"}
]

rows_with_all_features = pairwise.filter(
    pl.all_horizontal([pl.col(c).is_not_null() for c in feature_cols])
).height

pred_min = float(submission["Pred"].min())
pred_max = float(submission["Pred"].max())
invalid_id_count = submission.filter(~pl.col("ID").str.contains(r"^\d{4}_\d+_\d+$")).height

print("=== PIPELINE HEALTH DASHBOARD ===")
print(f"Map coverage          : {team_id_map_summary.get('match_rate', 'n/a')}")
print(f"Gold pass rate        : {gold_summary.get('quality_pass_rate', 'n/a')}")
print(f"Pairwise rows         : {pairwise.height:,}")
print(f"Feature columns       : {len(feature_cols)}")
print(f"Rows fully populated  : {rows_with_all_features:,} / {pairwise.height:,} ({rows_with_all_features / max(pairwise.height, 1):.1%})")
print(f"Submission rows       : {submission.height:,}")
print(f"Pred bounds           : [{pred_min:.4f}, {pred_max:.4f}]")
print(f"Malformed IDs         : {invalid_id_count}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sdiehl\\Desktop\\seandiehlprojects\\mm26\\artifacts\\latest\\reports\\team_id_map_summary.json'

---
## Part 1 — Bronze Layer: Kaggle Source Data

The Kaggle dataset provides the authoritative game results and team registry for both the men's and women's tournaments. The pipeline copies every CSV verbatim into parquet snapshots under `bronze/kaggle/` so they're immutable and fast to query.

**Key Kaggle files used downstream:**
- `MTeams` / `WTeams` — team IDs and names (the master registry)
- `MRegularSeasonDetailedResults` — men's regular season box scores from 2003 onward
- `WRegularSeasonDetailedResults` — women's regular season box scores from 1998 onward
- `MNCAATourneyCompactResults` / `WNCAATourneyCompactResults` — historical tournament outcomes
- `SampleSubmissionStage2` — defines the exact set of matchups we must predict

In [ ]:
# --- Bronze / Kaggle: Team registry ---
# MTeams is the authoritative list of men's Division-I programs.
# TeamID is the key that links every other dataset.
# FirstD1Season / LastD1Season tell us when a school was D1 (LastD1Season=2026 means currently active).
m_teams = pl.read_parquet(BASE / "bronze/kaggle/MTeams.parquet")
print(f"Men's teams: {m_teams.height} rows")
print(m_teams.head(10))

Men's teams: 381 rows
shape: (10, 4)
┌────────┬───────────────┬───────────────┬──────────────┐
│ TeamID ┆ TeamName      ┆ FirstD1Season ┆ LastD1Season │
│ ---    ┆ ---           ┆ ---           ┆ ---          │
│ i64    ┆ str           ┆ i64           ┆ i64          │
╞════════╪═══════════════╪═══════════════╪══════════════╡
│ 1101   ┆ Abilene Chr   ┆ 2014          ┆ 2026         │
│ 1102   ┆ Air Force     ┆ 1985          ┆ 2026         │
│ 1103   ┆ Akron         ┆ 1985          ┆ 2026         │
│ 1104   ┆ Alabama       ┆ 1985          ┆ 2026         │
│ 1105   ┆ Alabama A&M   ┆ 2000          ┆ 2026         │
│ 1106   ┆ Alabama St    ┆ 1985          ┆ 2026         │
│ 1107   ┆ SUNY Albany   ┆ 2000          ┆ 2026         │
│ 1108   ┆ Alcorn St     ┆ 1985          ┆ 2026         │
│ 1109   ┆ Alliant Intl  ┆ 1985          ┆ 1991         │
│ 1110   ┆ American Univ ┆ 1985          ┆ 2026         │
└────────┴───────────────┴───────────────┴──────────────┘


In [ ]:
# --- Bronze / Kaggle: Regular season detailed results ---
# This is the richest game-level dataset Kaggle provides.
# Each row is one game with full box-score totals (field goals, rebounds, etc.)
# The winner is always in the W* columns; loser in L* columns.
# DayNum is days since "Day 0" for that season — use MSeasons.DayZero to get calendar dates.
m_reg = pl.read_parquet(BASE / "bronze/kaggle/MRegularSeasonDetailedResults.parquet")
w_reg = pl.read_parquet(BASE / "bronze/kaggle/WRegularSeasonDetailedResults.parquet")
print(f"Men's detailed results: {m_reg.height:,} rows | Seasons: {m_reg['Season'].min()}–{m_reg['Season'].max()}")
print(f"Women's detailed results: {w_reg.height:,} rows | Seasons: {w_reg['Season'].min()}–{w_reg['Season'].max()}")
print()
print("Columns:", m_reg.columns)
m_reg.head(5)

Men's detailed results: 124,529 rows | Seasons: 2003–2026
Women's detailed results: 87,187 rows | Seasons: 2010–2026

Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']


Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2003,10,1104,68,1328,62,"""N""",0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
2003,10,1272,70,1393,63,"""N""",0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2003,11,1266,73,1437,61,"""N""",0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
2003,11,1296,56,1457,50,"""N""",0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
2003,11,1400,77,1208,71,"""N""",0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14


In [ ]:
# --- Bronze / Kaggle: NCAA Tournament results ---
# Compact results have only final scores. These are the ground-truth labels
# for every past tournament game and also define what we're predicting.
m_tourney = pl.read_parquet(BASE / "bronze/kaggle/MNCAATourneyCompactResults.parquet")
m_seeds   = pl.read_parquet(BASE / "bronze/kaggle/MNCAATourneySeeds.parquet")
sample    = pl.read_parquet(BASE / "bronze/kaggle/SampleSubmissionStage2.parquet")

print(f"Tournament results (M): {m_tourney.height:,} rows | Seasons: {m_tourney['Season'].min()}–{m_tourney['Season'].max()}")
print(f"Tournament seeds (M):   {m_seeds.height:,} rows")
print(f"Stage 2 submission rows to predict: {sample.height:,}")
print()
print("Sample submission format (ID = Season_TeamLow_TeamHigh, Pred = P(TeamLow wins))")
sample.head(5)

Tournament results (M): 2,585 rows | Seasons: 1985–2025
Tournament seeds (M):   2,694 rows
Stage 2 submission rows to predict: 132,133

Sample submission format (ID = Season_TeamLow_TeamHigh, Pred = P(TeamLow wins))


ID,Pred
str,f64
"""2026_1101_1102""",0.5
"""2026_1101_1103""",0.5
"""2026_1101_1104""",0.5
"""2026_1101_1105""",0.5
"""2026_1101_1106""",0.5


---
## Part 2 — Bronze Layer: CBBD API Data

The College Basketball Data API (CBBD) enriches the pipeline with three datasets that Kaggle doesn't provide:

| Dataset | Description | Grain |
|---|---|---|
| `games.parquet` | Game metadata: final scores, Elo ratings, seeds, excitement index | 1 row per game |
| `game_teams.parquet` | Team-level box scores per game (the "four factors" and more) | 2 rows per game (one per team) |
| `lines.parquet` | Raw betting lines per provider per game | 1 row per game × provider |

The pipeline fetches the **last 10 seasons (2016–2025)** using a **35-day sliding window** per season so it never hits the API's 3,000-result-per-call cap.

In [ ]:
# --- Bronze / CBBD: Games ---
# One row per game. Includes final scores, pre/post Elo for both teams,
# tournament seeds, site (neutral/home), conference flag, and an excitment index.
# `home_score` / `away_score` are the actual final points scored.
cbbd_games = pl.read_parquet(BASE / "bronze/cbbd/games.parquet")
print(f"CBBD games: {cbbd_games.height:,} rows  |  {cbbd_games['season'].min()}–{cbbd_games['season'].max()}")
print("Columns:", cbbd_games.columns)
cbbd_games.head(5)

CBBD games: 0 rows  |  None–None
Columns: ['game_id', 'season', 'season_type', 'status', 'start_date', 'home_team_id', 'home_team', 'home_conference', 'home_score', 'away_team_id', 'away_team', 'away_conference', 'away_score', 'neutral_site', 'conference_game', 'notes', 'home_elo_start', 'home_elo_end', 'away_elo_start', 'away_elo_end', 'home_seed', 'away_seed', 'excitement']


game_id,season,season_type,status,start_date,home_team_id,home_team,home_conference,home_score,away_team_id,away_team,away_conference,away_score,neutral_site,conference_game,notes,home_elo_start,home_elo_end,away_elo_start,away_elo_end,home_seed,away_seed,excitement
i64,i64,str,str,str,i64,str,str,f64,i64,str,str,f64,bool,bool,str,f64,f64,f64,f64,f64,f64,f64


In [ ]:
# --- Bronze / CBBD: Game Teams (box scores) ---
# Two rows per game — one per team. Contains the "four factors" and other advanced
# stats computed by CBBD: pace, offensive/defensive rating, field goal splits,
# rebounds, turnovers, free throw rate, etc.
# teamStats and opponentStats are flattened into columns like team_stats_points_total,
# team_stats_field_goals_made, team_stats_four_factors_effective_fg_pct, etc.
cbbd_gt = pl.read_parquet(BASE / "bronze/cbbd/game_teams.parquet")
print(f"CBBD game_teams: {cbbd_gt.height:,} rows  (2 rows per game = {cbbd_gt.height // 2:,} unique games)")
print(f"Columns ({len(cbbd_gt.columns)}):", cbbd_gt.columns[:16], "...")
cbbd_gt.select(["game_id", "season", "team", "opponent", "is_home", "pace",
                "game_type", "game_minutes"]).head(6)

CBBD game_teams: 0 rows  (2 rows per game = 0 unique games)
Columns (16): ['game_id', 'season', 'season_type', 'start_date', 'team_id', 'team', 'conference', 'opponent_id', 'opponent', 'opponent_conference', 'neutral_site', 'is_home', 'conference_game', 'game_type', 'game_minutes', 'pace'] ...


game_id,season,team,opponent,is_home,pace,game_type,game_minutes
i64,i64,str,str,bool,f64,str,f64


In [ ]:
# --- Bronze / CBBD: Lines ---
# One row per game × betting provider (e.g. "draftkings", "numberfire", "consensus").
# spread: negative = home team favoured (e.g. -3.5 means home favoured by 3.5 pts)
# spread_open: the opening line before market movement
# over_under: total projected points scored
# home_moneyline / away_moneyline: money line odds for each side
cbbd_lines = pl.read_parquet(BASE / "bronze/cbbd/lines.parquet")
print(f"CBBD lines: {cbbd_lines.height:,} rows")
print(f"Providers: {sorted(cbbd_lines['provider'].drop_nulls().unique().to_list())}")
print()
cbbd_lines.select(["game_id", "season", "home_team", "away_team",
                   "provider", "spread", "over_under", "home_moneyline"]).head(8)

CBBD lines: 0 rows
Providers: []



game_id,season,home_team,away_team,provider,spread,over_under,home_moneyline
i64,i64,str,str,str,f64,f64,f64


In [ ]:
# --- Lines quality check: verify lines match expected games ---
# 1. Every lines row should reference a game_id that exists in games.parquet.
#    Orphaned lines (no matching game) indicate a data integrity issue.
# 2. Lines are at grain: game × provider. Dividing by unique providers gives
#    an expected upper bound on unique games with lines.
# 3. Coverage = what % of games in the games table have at least one line.

all_game_ids = set(cbbd_games["game_id"].to_list())
line_game_ids = set(cbbd_lines["game_id"].drop_nulls().to_list())

games_with_lines = line_game_ids & all_game_ids  # matched
orphaned_lines = line_game_ids - all_game_ids  # lines with no game record

total_lines = cbbd_lines.height
unique_games_in_lines = cbbd_lines["game_id"].drop_nulls().n_unique()
unique_providers_in_lines = cbbd_lines["provider"].drop_nulls().n_unique()

total_games = len(all_game_ids)
lines_coverage_pct = (100 * len(games_with_lines) / total_games) if total_games else 0.0

print("=== Lines ↔ Games integrity check ===")
print(f"Total line rows             : {total_lines:,}")
print(f"Unique game IDs in lines    : {unique_games_in_lines:,}")
print(f"Unique providers            : {unique_providers_in_lines}")
print(f"Total games in games table  : {total_games:,}")
print(f"Games WITH at least one line: {len(games_with_lines):,}  ({lines_coverage_pct:.1f}% of all games)")
print(f"Games WITHOUT any line      : {total_games - len(games_with_lines):,}")
print(f"Orphaned lines (no game)    : {len(orphaned_lines):,}", end="")
print("  ✓ OK" if len(orphaned_lines) == 0 else "  ⚠ UNEXPECTED — investigate these game IDs")

print()
# Expected lines per game ≃ unique_games_in_lines × avg providers per game
provider_counts = cbbd_lines.group_by("game_id").agg(
    pl.col("provider").drop_nulls().n_unique().alias("provider_count")
)
avg_providers = provider_counts["provider_count"].mean() if provider_counts.height else 0.0
expected_total_rows = int(unique_games_in_lines * avg_providers) if avg_providers is not None else 0
print(f"Avg providers per game      : {float(avg_providers or 0.0):.2f}")
print(f"Expected total rows (≈)     : {unique_games_in_lines:,} games × {float(avg_providers or 0.0):.2f} providers ≈ {expected_total_rows:,}  (actual: {total_lines:,})")

print()
# Null spread / over_under rates — high nulls = providers that only supply partial data
null_spread = cbbd_lines["spread"].is_null().sum()
null_ou = cbbd_lines["over_under"].is_null().sum()
spread_null_pct = (100 * null_spread / total_lines) if total_lines else 0.0
ou_null_pct = (100 * null_ou / total_lines) if total_lines else 0.0
print(f"Null spread values          : {null_spread:,} / {total_lines:,}  ({spread_null_pct:.1f}%)")
print(f"Null over_under values      : {null_ou:,}  / {total_lines:,}  ({ou_null_pct:.1f}%)")

=== Lines ↔ Games integrity check ===
Total line rows             : 0
Unique game IDs in lines    : 0
Unique providers            : 0
Total games in games table  : 0
Games WITH at least one line: 0  (0.0% of all games)
Games WITHOUT any line      : 0
Orphaned lines (no game)    : 0  ✓ OK

Avg providers per game      : 0.00
Expected total rows (≈)     : 0 games × 0.00 providers ≈ 0  (actual: 0)

Null spread values          : 0 / 0  (0.0%)
Null over_under values      : 0  / 0  (0.0%)


---
## Part 3 — Silver Layer: Cleaned & Joined Data

The silver layer transforms the raw bronze data into a clean, analysis-ready format.

### What happens in the transform step:

1. **`game_fact.parquet`** — Kaggle detailed results are "exploded" from winner/loser format into one row per team per game (so each game becomes 2 rows). This makes it easy to compute per-team season stats.

2. **`team_dim.parquet`** — A dimension table of all teams (men's and women's) with their metadata.

3. **`team_id_map.parquet`** — CBBD uses its own team name strings; this table maps CBBD team names → Kaggle TeamIDs by fuzzy name matching. Coverage report shows how many were matched.

4. **`cbbd_games.parquet`** — CBBD games enriched with mapped Kaggle TeamIDs and a canonical `team_low / team_high` pair key.

5. **`cbbd_game_fact.parquet`** — The CBBD box score data (from `game_teams`), one row per team per game.

6. **`cbbd_line_consensus.parquet`** — Provider lines aggregated to a single consensus spread per game using the median across non-null providers.

7. **`cbbd_lines.parquet`** — All raw provider lines with Kaggle TeamIDs mapped in.

8. **`quality_flags.parquet`** — A pass/fail quality flag per row indicating whether a game has all required features for model training.

In [ ]:
# --- Silver: game_fact ---
# One row per team per game. The Kaggle detailed results are in winner/loser format.
# The pipeline "flips" them so every team appears on both sides of each game.
# sex = 'M' or 'W', team_loc = 'H'(ome) / 'A'(way) / 'N'(eutral)
# win = 1 if this team won, 0 if they lost
# game_key = f"{season}_{team_low}_{team_high}" — a stable game identifier
game_fact = pl.read_parquet(BASE / "silver/game_fact.parquet")
print(f"game_fact: {game_fact.height:,} rows — {game_fact.height // 2:,} unique games")
print(f"Seasons: {game_fact['season'].min()}–{game_fact['season'].max()}")
print()
game_fact.head(6)

game_fact: 423,432 rows — 211,716 unique games
Seasons: 2003–2026



sex,season,day_num,team_id,opp_team_id,team_score,opp_score,team_loc,num_ot,win,fgm,fga,fgm3,fga3,ftm,fta,oreb,dreb,ast,tov,stl,blk,pf,opp_fgm,opp_fga,opp_fgm3,opp_fga3,opp_ftm,opp_fta,opp_oreb,opp_dreb,opp_ast,opp_tov,opp_stl,opp_blk,opp_pf,team_low,team_high,game_key
str,i64,i64,i64,i64,f64,f64,str,i64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,str
"""M""",2003,10,1104,1328,68.0,62.0,"""N""",0,1,27.0,58.0,3.0,14.0,11.0,18.0,14.0,24.0,13.0,23.0,7.0,1.0,22.0,22.0,53.0,2.0,10.0,16.0,22.0,10.0,22.0,8.0,18.0,9.0,2.0,20.0,1104,1328,"""M_2003_10_1104_1328"""
"""M""",2003,10,1272,1393,70.0,63.0,"""N""",0,1,26.0,62.0,8.0,20.0,10.0,19.0,15.0,28.0,16.0,13.0,4.0,4.0,18.0,24.0,67.0,6.0,24.0,9.0,20.0,20.0,25.0,7.0,12.0,8.0,6.0,16.0,1272,1393,"""M_2003_10_1272_1393"""
"""M""",2003,11,1266,1437,73.0,61.0,"""N""",0,1,24.0,58.0,8.0,18.0,17.0,29.0,17.0,26.0,15.0,10.0,5.0,2.0,25.0,22.0,73.0,3.0,26.0,14.0,23.0,31.0,22.0,9.0,12.0,2.0,5.0,23.0,1266,1437,"""M_2003_11_1266_1437"""
"""M""",2003,11,1296,1457,56.0,50.0,"""N""",0,1,18.0,38.0,3.0,9.0,17.0,31.0,6.0,19.0,11.0,12.0,14.0,2.0,18.0,18.0,49.0,6.0,22.0,8.0,15.0,17.0,20.0,9.0,19.0,4.0,3.0,23.0,1296,1457,"""M_2003_11_1296_1457"""
"""M""",2003,11,1400,1208,77.0,71.0,"""N""",0,1,30.0,61.0,6.0,14.0,11.0,13.0,17.0,22.0,12.0,14.0,4.0,4.0,20.0,24.0,62.0,6.0,16.0,17.0,27.0,21.0,15.0,12.0,10.0,7.0,1.0,14.0,1208,1400,"""M_2003_11_1208_1400"""
"""M""",2003,11,1458,1186,81.0,55.0,"""H""",0,1,26.0,57.0,6.0,12.0,23.0,27.0,12.0,24.0,12.0,9.0,9.0,3.0,18.0,20.0,46.0,3.0,11.0,12.0,17.0,6.0,22.0,8.0,19.0,4.0,3.0,25.0,1186,1458,"""M_2003_11_1186_1458"""


In [ ]:
# --- Silver: team_dim ---
# Dimension table for all teams. Links sex + team_id to a team name.
# first_d1_season / last_d1_season indicate D1 membership windows.
team_dim = pl.read_parquet(BASE / "silver/team_dim.parquet")
print(f"team_dim: {team_dim.height} teams  (M: {team_dim.filter(pl.col('sex')=='M').height}, W: {team_dim.filter(pl.col('sex')=='W').height})")
team_dim.head(8)

team_dim: 760 teams  (M: 381, W: 379)


sex,team_id,team_name,first_d1_season,last_d1_season
str,i64,str,i64,i64
"""M""",1101,"""Abilene Chr""",2014,2026
"""M""",1102,"""Air Force""",1985,2026
"""M""",1103,"""Akron""",1985,2026
"""M""",1104,"""Alabama""",1985,2026
"""M""",1105,"""Alabama A&M""",2000,2026
"""M""",1106,"""Alabama St""",1985,2026
"""M""",1107,"""SUNY Albany""",2000,2026
"""M""",1108,"""Alcorn St""",1985,2026


In [ ]:
# --- Silver: team_id_map (CBBD → Kaggle) ---

# The CBBD API uses its own team names; Kaggle uses numeric TeamIDs.

# The pipeline normalises both sides (lowercase, remove punctuation, expand "&" → "and")

# and performs exact string matching, then fuzzy matching for near-misses.

# mapped=True means a Kaggle TeamID was found; mapped=False = unmatched.

team_id_map = pl.read_parquet(BASE / "silver/team_id_map.parquet")

matched   = team_id_map.filter(pl.col("mapped") == True)

unmatched = team_id_map.filter(pl.col("mapped") == False)



total_map_rows = team_id_map.height

coverage_pct = (100 * matched.height / total_map_rows) if total_map_rows else 0.0



print(f"Mapping coverage: {matched.height} / {total_map_rows} matched  ({coverage_pct:.1f}%)")

print(f"Unmatched CBBD team names ({unmatched.height}):")

print(unmatched["cbbd_team_name"].to_list()[:20])

Mapping coverage: 0 / 0 matched  (0.0%)
Unmatched CBBD team names (0):
[]


In [ ]:
# --- Silver: cbbd_line_consensus ---
# Raw lines from multiple providers are aggregated to a single consensus per game.
# consensus_home_spread = median of all non-null provider spreads
# provider_count = how many providers contributed data for this game
# Games with only 1 provider are less reliable than games with 3+.
cbbd_consensus = pl.read_parquet(BASE / "silver/cbbd_line_consensus.parquet")
print(f"cbbd_line_consensus: {cbbd_consensus.height:,} rows")
print("Columns:", cbbd_consensus.columns)
print()
# Provider coverage distribution
provider_dist = cbbd_consensus.group_by("provider_count").agg(pl.len().alias("games")).sort("provider_count")
print("Provider count distribution:")
print(provider_dist)
cbbd_consensus.head(5)

cbbd_line_consensus: 0 rows
Columns: ['game_id', 'season', 'start_date', 'home_team_id', 'away_team_id', 'consensus_home_spread', 'consensus_home_spread_open', 'consensus_over_under', 'provider_count', 'kaggle_home_team_id', 'kaggle_away_team_id', 'team_low', 'team_high', 'consensus_low_spread']

Provider count distribution:
shape: (0, 2)
┌────────────────┬───────┐
│ provider_count ┆ games │
│ ---            ┆ ---   │
│ i64            ┆ u32   │
╞════════════════╪═══════╡
└────────────────┴───────┘


game_id,season,start_date,home_team_id,away_team_id,consensus_home_spread,consensus_home_spread_open,consensus_over_under,provider_count,kaggle_home_team_id,kaggle_away_team_id,team_low,team_high,consensus_low_spread
i64,i64,str,i64,i64,f64,f64,f64,i64,i64,i64,i64,i64,f64


In [ ]:
# --- Silver: cbbd_game_fact (box score per team) ---
# One row per team per game pulled from the CBBD game_teams endpoint.
# Contains pace (possessions per 40 mins), plus nested teamStats/opponentStats
# flattened into columns: team_stats_field_goals_made, team_stats_four_factors_effective_fg_pct, etc.
cbbd_gf = pl.read_parquet(BASE / "silver/cbbd_game_fact.parquet")
print(f"cbbd_game_fact: {cbbd_gf.height:,} rows  ({cbbd_gf.height // 2:,} unique games)")
print(f"Columns ({len(cbbd_gf.columns)}): {cbbd_gf.columns}")
# Show a focused slice of the box-score stats
stat_cols = [c for c in cbbd_gf.columns if "team_stats" in c]
print(f"\nBox score stat columns ({len(stat_cols)}):")
print(stat_cols)

cbbd_game_fact: 0 rows  (0 unique games)
Columns (16): ['game_id', 'season', 'season_type', 'start_date', 'team_id', 'team', 'conference', 'opponent_id', 'opponent', 'opponent_conference', 'neutral_site', 'is_home', 'conference_game', 'game_type', 'game_minutes', 'pace']

Box score stat columns (0):
[]


In [ ]:
# --- Silver: quality_flags ---


# Every row in game_fact gets a row_quality_pass flag.

# A row passes if all features needed for model training are non-null.

# The train/holdout split only uses rows where row_quality_pass = True.

quality = pl.read_parquet(BASE / "silver/quality_flags.parquet")

passing_rows = quality.filter(pl.col("row_quality_pass")).height

total_quality_rows = quality.height

pass_rate = (passing_rows / total_quality_rows) if total_quality_rows else 0.0



print(f"Quality flags: {total_quality_rows:,} total rows")

print(f"Pass rate: {pass_rate:.1%}  ({passing_rows:,} passing)")

print()

# Break down by sex and season

by_season = (

    quality

    .group_by(["sex", "season"])

    .agg(

        pl.len().alias("total_rows"),

        pl.col("row_quality_pass").sum().alias("passing"),

    )

    .with_columns(

        pl.when(pl.col("total_rows") > 0)

        .then((pl.col("passing") / pl.col("total_rows") * 100).round(1))

        .otherwise(0.0)

        .alias("pass_pct")

    )

    .sort(["sex", "season"])

)

print(by_season)

Quality flags: 423,432 total rows
Pass rate: 100.0%  (423,432 passing)

shape: (41, 5)
┌─────┬────────┬────────────┬─────────┬──────────┐
│ sex ┆ season ┆ total_rows ┆ passing ┆ pass_pct │
│ --- ┆ ---    ┆ ---        ┆ ---     ┆ ---      │
│ str ┆ i64    ┆ u32        ┆ u32     ┆ f64      │
╞═════╪════════╪════════════╪═════════╪══════════╡
│ M   ┆ 2003   ┆ 9232       ┆ 9232    ┆ 100.0    │
│ M   ┆ 2004   ┆ 9142       ┆ 9142    ┆ 100.0    │
│ M   ┆ 2005   ┆ 9350       ┆ 9350    ┆ 100.0    │
│ M   ┆ 2006   ┆ 9514       ┆ 9514    ┆ 100.0    │
│ M   ┆ 2007   ┆ 10086      ┆ 10086   ┆ 100.0    │
│ …   ┆ …      ┆ …          ┆ …       ┆ …        │
│ W   ┆ 2022   ┆ 10120      ┆ 10120   ┆ 100.0    │
│ W   ┆ 2023   ┆ 10748      ┆ 10748   ┆ 100.0    │
│ W   ┆ 2024   ┆ 10828      ┆ 10828   ┆ 100.0    │
│ W   ┆ 2025   ┆ 10888      ┆ 10888   ┆ 100.0    │
│ W   ┆ 2026   ┆ 10958      ┆ 10958   ┆ 100.0    │
└─────┴────────┴────────────┴─────────┴──────────┘


---
## Part 4 — Gold Layer: Model-Ready Features

The gold layer aggregates game_fact into features suitable for machine learning. There are three core tables:

| Table | Grain | Purpose |
|---|---|---|
| `team_season_features` | 1 row per team per season | Season-level averages: PPG, win rate, margin, ELO, heat, box scores, SOS |
| `game_features` | 1 row per game | Aggregated stats for each side of a historical game |
| `pairwise_features` | 1 row per matchup in submission | The direct model input: pairwise feature differences for `team_low` vs `team_high` |

The exact number of model features is read from the latest model report and can change as feature work evolves.

**Train/validate/predict split:**
- **Train**: seasons 2003-2024
- **Holdout/validation**: season 2025
- **Prediction target**: season 2026

In [ ]:
# --- Gold: team_season_features ---
# One row per team per season. The pipeline computes these from the silver game_fact:
#   games_played, wins, losses
#   avg_pts_for, avg_pts_against, avg_margin (offensive/defensive efficiency proxies)
#   avg_num_ot (overtime rate — higher = more close games)
#   home_win_rate, away_win_rate (location-adjusted performance)
team_feats = pl.read_parquet(BASE / "gold/team_season_features.parquet")
print(f"team_season_features: {team_feats.height:,} rows")
print(f"Seasons: {team_feats['season'].min()}–{team_feats['season'].max()}")
print("Columns:", team_feats.columns)
print()
# Example: top 10 men's teams by average margin in 2025
(
    team_feats
    .filter((pl.col("season") == 2025) & (pl.col("sex") == "M"))
    .sort("avg_margin", descending=True)
    .join(team_dim.filter(pl.col("sex") == "M").select(["team_id", "team_name"]), on="team_id", how="left")
    .select(["team_name", "season", "games_played", "wins", "avg_pts_for", "avg_pts_against", "avg_margin"])
    .head(10)
)

team_season_features: 14,311 rows
Seasons: 2003–2026
Columns: ['sex', 'season', 'team_id', 'games_played', 'wins', 'losses', 'avg_pts_for', 'avg_pts_against', 'avg_margin', 'avg_num_ot', 'last5_avg_pts_for', 'last5_avg_pts_against', 'last5_avg_margin', 'last5_win_rate', 'fg_pct', 'fg3_pct', 'ft_pct', 'opp_fg_pct', 'avg_oreb', 'avg_dreb', 'avg_ast', 'avg_tov', 'avg_stl', 'avg_blk', 'avg_opp_tov', 'avg_possessions', 'last5_fg_pct', 'win_rate', 'avg_reb_margin', 'ast_to_ratio', 'avg_stl_blk', 'season_end_elo', 'sos', 'pre_tourney_heat_1g', 'pre_tourney_heat_3g', 'pre_tourney_heat_5g', 'pre_tourney_sw_heat_1g', 'pre_tourney_sw_heat_3g', 'pre_tourney_sw_heat_5g', 'pre_tourney_heat_volatility_5g', 'heat_trend', 'abs_heat_5g', 'quality']



team_name,season,games_played,wins,avg_pts_for,avg_pts_against,avg_margin
str,i64,u32,i32,f64,f64,f64
"""Duke""",2025,34,31,82.705882,61.911765,20.794118
"""Gonzaga""",2025,33,25,86.636364,69.636364,17.0
"""Florida""",2025,34,30,85.411765,69.235294,16.176471
"""Houston""",2025,34,30,74.205882,58.470588,15.735294
"""UC San Diego""",2025,32,28,77.9375,62.84375,15.09375
"""Maryland""",2025,33,25,81.666667,67.0,14.666667
"""Auburn""",2025,33,28,83.848485,69.606061,14.242424
"""VCU""",2025,33,27,76.333333,62.515152,13.818182
"""Texas Tech""",2025,33,25,80.909091,67.575758,13.333333


In [ ]:
# --- Gold: pairwise_features ---

# This is the direct input to the model. One row per matchup in the submission file.

# Each row represents a potential tournament game: team_low (lower TeamID) vs team_high.

# The features are *differences* between the two teams' season stats, so positive values

# mean team_low has the advantage on that dimension.

# Feature columns that end in _diff = team_low_stat - team_high_stat.

# The target is: did team_low win? (1 = yes, 0 = no)

pairs = pl.read_parquet(BASE / "gold/pairwise_features.parquet")

print(f"pairwise_features: {pairs.height:,} rows  (= submission rows to predict)")

print("Columns:", pairs.columns)

print()

# Check how many have all features vs missing features

feature_cols = [c for c in pairs.columns if c not in ("ID", "season", "team_low", "team_high", "sex", "label")]

null_counts = pairs.select([pl.col(c).is_null().sum().alias(c) for c in feature_cols])

complete = pairs.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in feature_cols])).height

total_pairs = pairs.height

complete_pct = (100 * complete / total_pairs) if total_pairs else 0.0

print(f"Rows with all features populated: {complete:,} / {total_pairs:,}  ({complete_pct:.1f}%)")

pairs.head(3)

pairwise_features: 132,133 rows  (= submission rows to predict)
Columns: ['ID', 'season', 'team_low', 'team_high', 'sex', 'games_played_low', 'wins_low', 'losses_low', 'avg_pts_for_low', 'avg_pts_against_low', 'avg_margin_low', 'avg_num_ot_low', 'last5_avg_pts_for_low', 'last5_avg_pts_against_low', 'last5_avg_margin_low', 'last5_win_rate_low', 'fg_pct_low', 'fg3_pct_low', 'ft_pct_low', 'opp_fg_pct_low', 'avg_oreb_low', 'avg_dreb_low', 'avg_ast_low', 'avg_tov_low', 'avg_stl_low', 'avg_blk_low', 'avg_opp_tov_low', 'avg_possessions_low', 'last5_fg_pct_low', 'win_rate_low', 'avg_reb_margin_low', 'ast_to_ratio_low', 'avg_stl_blk_low', 'season_end_elo_low', 'sos_low', 'pre_tourney_heat_1g_low', 'pre_tourney_heat_3g_low', 'pre_tourney_heat_5g_low', 'pre_tourney_sw_heat_1g_low', 'pre_tourney_sw_heat_3g_low', 'pre_tourney_sw_heat_5g_low', 'pre_tourney_heat_volatility_5g_low', 'heat_trend_low', 'abs_heat_5g_low', 'quality_low', 'games_played_high', 'wins_high', 'losses_high', 'avg_pts_for_high',

ID,season,team_low,team_high,sex,games_played_low,wins_low,losses_low,avg_pts_for_low,avg_pts_against_low,avg_margin_low,avg_num_ot_low,last5_avg_pts_for_low,last5_avg_pts_against_low,last5_avg_margin_low,last5_win_rate_low,fg_pct_low,fg3_pct_low,ft_pct_low,opp_fg_pct_low,avg_oreb_low,avg_dreb_low,avg_ast_low,avg_tov_low,avg_stl_low,avg_blk_low,avg_opp_tov_low,avg_possessions_low,last5_fg_pct_low,win_rate_low,avg_reb_margin_low,ast_to_ratio_low,avg_stl_blk_low,season_end_elo_low,sos_low,pre_tourney_heat_1g_low,pre_tourney_heat_3g_low,…,pre_tourney_heat_volatility_5g_high,heat_trend_high,abs_heat_5g_high,quality_high,consensus_low_spread,consensus_over_under,provider_count,seed_low,seed_high,diff_win_rate,diff_avg_margin,diff_avg_pts_for,diff_defense_proxy,diff_last5_win_rate,diff_last5_avg_margin,diff_elo,diff_heat_1g,diff_heat_3g,diff_heat_5g,diff_seed,diff_heat_trend,diff_abs_heat_5g,diff_sw_heat_5g,diff_heat_volatility,diff_sos,diff_fg_pct,diff_fg3_pct,diff_ft_pct,diff_opp_fg_pct,diff_reb_margin,diff_ast_to_ratio,diff_stl_blk,diff_possessions,seed_product,consensus_low_spread_filled,diff_quality,diff_quality_minus_elo
str,i64,i64,i64,str,u32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2026_1101_1102""",2026,1101,1102,"""M""",30,11,19,67.933333,73.866667,-5.933333,0.033333,66.0,76.6,-10.6,0.2,0.424039,0.310409,0.704246,0.50342,8.366667,18.3,12.533333,13.533333,9.6,2.566667,15.466667,70.614167,0.418699,0.366667,-1.8,0.926108,12.166667,1387.599744,1515.465056,6.02686,-7.60551,…,2.327939,-7.044956,0.7659,-14.980516,null,null,0,null,null,0.272917,12.410417,6.402083,6.008333,0.2,6.6,229.708395,-1.783997,-13.898166,-0.80324,0.0,0.980757,-0.728561,0.921104,3.041734,-46.009419,-0.001129,0.004853,0.063131,0.012689,4.7625,0.021457,4.166667,3.312604,64.0,0.0,9.295657,6.998573
"""2026_1101_1103""",2026,1101,1103,"""M""",30,11,19,67.933333,73.866667,-5.933333,0.033333,66.0,76.6,-10.6,0.2,0.424039,0.310409,0.704246,0.50342,8.366667,18.3,12.533333,13.533333,9.6,2.566667,15.466667,70.614167,0.418699,0.366667,-1.8,0.926108,12.166667,1387.599744,1515.465056,6.02686,-7.60551,…,9.694293,5.987051,3.266243,9.654835,null,null,0,null,12,-0.477083,-18.308333,-18.941667,0.633333,-0.8,-23.6,-415.94261,8.747668,-6.715457,-3.303582,-4.0,-12.051251,-3.228903,-3.793366,-4.32462,37.071138,-0.073714,-0.074547,-0.048331,0.073595,-6.925,-0.771221,2.041667,-1.587396,96.0,0.0,-15.339694,-11.180268
"""2026_1101_1104""",2026,1101,1104,"""M""",30,11,19,67.933333,73.866667,-5.933333,0.033333,66.0,76.6,-10.6,0.2,0.424039,0.310409,0.704246,0.50342,8.366667,18.3,12.533333,13.533333,9.6,2.566667,15.466667,70.614167,0.418699,0.366667,-1.8,0.926108,12.166667,1387.599744,1515.465056,6.02686,-7.60551,…,6.865783,3.258247,1.419334,18.550117,null,null,0,null,4,-0.352083,-14.183333,-23.785417,9.602083,-0.4,-16.2,-374.451891,10.704441,-4.125616,1.381994,4.0,-9.322447,-1.381994,1.16881,-1.49611,-125.196301,-0.033741,-0.047681,-0.06106,0.070313,-2.45625,-0.782501,0.666667,-6.148333,32.0,0.0,-24.234976,-20.490457


In [ ]:
# --- Gold: game_features ---
# Aggregated features at the game level (used to build pairwise_features).
# avg_team_points / avg_opp_points are rolling season averages at the time of the game.
game_feats = pl.read_parquet(BASE / "gold/game_features.parquet")
print(f"game_features: {game_feats.height:,} rows")
print("Columns:", game_feats.columns)
game_feats.head(5)

game_features: 211,716 rows
Columns: ['sex', 'season', 'game_key', 'avg_team_points', 'avg_opp_points', 'sum_win_rows']


sex,season,game_key,avg_team_points,avg_opp_points,sum_win_rows
str,i64,str,f64,f64,i32
"""M""",2003,"""M_2003_82_1148_1192""",62.5,62.5,1
"""M""",2008,"""M_2008_104_1130_1438""",76.5,76.5,1
"""M""",2011,"""M_2011_73_1148_1357""",81.5,81.5,1
"""W""",2024,"""W_2024_54_3417_3425""",67.5,67.5,1
"""M""",2003,"""M_2003_108_1205_1359""",58.5,58.5,1


---
## Part 5 — CBBD Coverage & Mapping Quality

Two reports from the pipeline tell us how well the CBBD data integrates with the Kaggle data:
- **`cbbd_coverage_summary.json`** — how many games and teams were captured per season
- **`team_id_map_summary.json`** — what fraction of CBBD team names were successfully matched to Kaggle TeamIDs

In [ ]:
# --- CBBD coverage by season ---
coverage = json.loads((BASE / "reports/cbbd_coverage_summary.json").read_text(encoding="utf-8"))
map_summary = json.loads((BASE / "reports/team_id_map_summary.json").read_text(encoding="utf-8"))

print("Team ID mapping summary:")
for k, v in map_summary.items():
    print(f"  {k}: {v}")

print()
coverage_df = pl.DataFrame(coverage["seasons"])
print("CBBD coverage per season:")
print(coverage_df)

Team ID mapping summary:
  total: 0
  mapped: 0
  unmapped: 0
  mapped_pct: 0.0

CBBD coverage per season:
shape: (0, 0)
┌┐
╞╡
└┘


---
## Part 6 — Submission Output

The final output is `submission.csv` in Kaggle-required format:
- `ID` = `Season_TeamLow_TeamHigh` (team with lower TeamID first)
- `Pred` = probability that `TeamLow` beats `TeamHigh`

The current pipeline writes model-only calibrated probabilities and applies seed-aware clipping bounds.

This section validates the submission contract:
- row count is non-zero
- IDs match expected pattern
- `Pred` has no null values
- `Pred` falls within clipped probability bounds

In [ ]:
# --- Submission file contract checks ---
submission = pl.read_csv(BASE / "submission.csv")

null_pred = submission.filter(pl.col("Pred").is_null()).height
bad_id = submission.filter(~pl.col("ID").str.contains(r"^\d{4}_\d+_\d+$")).height
below_min = submission.filter(pl.col("Pred") < 0.0).height
above_max = submission.filter(pl.col("Pred") > 1.0).height

print(f"Submission rows: {submission.height:,}")
print(f"Null Pred rows: {null_pred}")
print(f"Malformed IDs : {bad_id}")
print(f"Pred < 0 rows : {below_min}")
print(f"Pred > 1 rows : {above_max}")
print(f"Pred range    : [{submission['Pred'].min():.4f}, {submission['Pred'].max():.4f}]")
print()
print("Prediction distribution (sanity check):")
print(submission["Pred"].describe())
print()
print("Sample rows:")
submission.head(10)

Submission rows: 132,133
Null Pred rows: 0
Malformed IDs : 0
Pred < 0 rows : 0
Pred > 1 rows : 0
Pred range    : [0.0100, 0.9900]

Prediction distribution (sanity check):
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 132133.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 0.490078 │
│ std        ┆ 0.243764 │
│ min        ┆ 0.01     │
│ 25%        ┆ 0.25     │
│ 50%        ┆ 0.57732  │
│ 75%        ┆ 0.666667 │
│ max        ┆ 0.99     │
└────────────┴──────────┘

Sample rows:


ID,Pred
str,f64
"""2026_1101_1102""",0.731343
"""2026_1101_1103""",0.25
"""2026_1101_1104""",0.25
"""2026_1101_1105""",0.613636
"""2026_1101_1106""",0.66685
"""2026_1101_1107""",0.613636
"""2026_1101_1108""",0.731343
"""2026_1101_1110""",0.344444
"""2026_1101_1111""",0.344444


In [ ]:
# --- Holdout validation: check model performance on 2025 tournament ---
# The pipeline holds out 2025 and evaluates log-loss on known tournament outcomes.
gold_summary = json.loads((BASE / "reports/gold_quality_summary.json").read_text(encoding="utf-8"))
print("Gold quality summary:")
for k, v in gold_summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Gold quality summary:
  total_rows: 132133
  good_rows: 132133
  bad_rows: 0


---
## Part 7 — End-to-End Trace: One Matchup

To make the pipeline concrete, let's trace a single team pairing all the way from raw data through to its final submission prediction.

In [ ]:
# Pick the first submission row and trace it back through every layer
example_row = submission.head(1)
example_id = example_row["ID"][0]
season_str, t_low_str, t_high_str = example_id.split("_")
season = int(season_str)
t_low = int(t_low_str)
t_high = int(t_high_str)
pred = example_row["Pred"][0]

# Look up team names
def team_name(tid):
    row = team_dim.filter(pl.col("team_id") == tid)
    return row["team_name"][0] if row.height > 0 else str(tid)

name_low  = team_name(t_low)
name_high = team_name(t_high)

print(f"Matchup ID     : {example_id}")
print(f"Season         : {season}")
print(f"Team low  ({t_low}): {name_low}")
print(f"Team high ({t_high}): {name_high}")
print(f"Prediction     : {pred:.4f}  → {name_low} has a {pred:.1%} chance of winning")

Matchup ID     : 2026_1101_1102
Season         : 2026
Team low  (1101): Abilene Chr
Team high (1102): Air Force
Prediction     : 0.7313  → Abilene Chr has a 73.1% chance of winning


In [ ]:
# Step 1 of trace — Silver: what were each team's season stats?
print(f"=== {name_low} ({t_low}) — Season {season} ===")
print(team_feats.filter((pl.col("team_id") == t_low) & (pl.col("season") == season)))

print(f"\n=== {name_high} ({t_high}) — Season {season} ===")
print(team_feats.filter((pl.col("team_id") == t_high) & (pl.col("season") == season)))

=== Abilene Chr (1101) — Season 2026 ===
shape: (1, 43)
┌─────┬────────┬─────────┬──────────────┬───┬───────────────┬────────────┬─────────────┬───────────┐
│ sex ┆ season ┆ team_id ┆ games_played ┆ … ┆ pre_tourney_h ┆ heat_trend ┆ abs_heat_5g ┆ quality   │
│ --- ┆ ---    ┆ ---     ┆ ---          ┆   ┆ eat_volatilit ┆ ---        ┆ ---         ┆ ---       │
│ str ┆ i64    ┆ i64     ┆ u32          ┆   ┆ y_5g          ┆ f64        ┆ f64         ┆ f64       │
│     ┆        ┆         ┆              ┆   ┆ ---           ┆            ┆             ┆           │
│     ┆        ┆         ┆              ┆   ┆ f64           ┆            ┆             ┆           │
╞═════╪════════╪═════════╪══════════════╪═══╪═══════════════╪════════════╪═════════════╪═══════════╡
│ M   ┆ 2026   ┆ 1101    ┆ 30           ┆ … ┆ 5.369673      ┆ -6.064199  ┆ 0.03734     ┆ -5.684859 │
└─────┴────────┴─────────┴──────────────┴───┴───────────────┴────────────┴─────────────┴───────────┘

=== Air Force (1102) — Season 2026

In [ ]:
# Step 2 of trace — Gold: what does the pairwise feature vector look like?
pair_row = pairs.filter(
    (pl.col("team_low") == t_low) & (pl.col("team_high") == t_high) & (pl.col("season") == season)
)
print(f"Pairwise feature row for {name_low} vs {name_high} ({season}):")
print(pair_row.to_pandas().T.to_string())

Pairwise feature row for Abilene Chr vs Air Force (2026):
                                                  0
ID                                   2026_1101_1102
season                                         2026
team_low                                       1101
team_high                                      1102
sex                                               M
games_played_low                                 30
wins_low                                         11
losses_low                                       19
avg_pts_for_low                           67.933333
avg_pts_against_low                       73.866667
avg_margin_low                            -5.933333
avg_num_ot_low                             0.033333
last5_avg_pts_for_low                          66.0
last5_avg_pts_against_low                      76.6
last5_avg_margin_low                          -10.6
last5_win_rate_low                              0.2
fg_pct_low                                 0.424039
fg3_pc

In [ ]:
# Step 3 of trace — Were there any betting lines for this team pair?
lines_for_pair = cbbd_consensus.filter(
    (pl.col("season") == season) &
    (
        ((pl.col("home_team_id").cast(pl.Int64) == t_low) | (pl.col("away_team_id").cast(pl.Int64) == t_low)) &
        ((pl.col("home_team_id").cast(pl.Int64) == t_high) | (pl.col("away_team_id").cast(pl.Int64) == t_high))
    )
)
if lines_for_pair.height > 0:
    print(f"Betting lines for {name_low} vs {name_high} in {season}:")
    print(lines_for_pair.select(["game_id", "start_date", "consensus_home_spread",
                                  "consensus_over_under", "provider_count"]))
else:
    print(f"No betting lines found for this specific matchup in {season}.")

No betting lines found for this specific matchup in 2026.


---
## Part 8 — Quick Reference: Column Glossary

| Column | Table | Meaning |
|---|---|---|
| `sex` | most | `'M'` = men's, `'W'` = women's |
| `season` | most | The year the season ended (e.g. 2026 = 2025–26 season) |
| `team_id` / `opp_team_id` | game_fact | Kaggle TeamID (men's 1000–1999, women's 3000–3999) |
| `team_low` / `team_high` | pairwise_features | Canonical matchup key: lower TeamID vs higher TeamID |
| `day_num` | game_fact | Days since Day 0 of that season (see MSeasons for calendar dates) |
| `win` | game_fact | 1 if this team won, 0 if they lost |
| `game_key` | game_fact, game_features | `season_teamlow_teamhigh` — stable cross-table join key |
| `row_quality_pass` | quality_flags | True if all required features are non-null for model training |
| `consensus_home_spread` | cbbd_line_consensus | Median spread across providers (negative = home favoured) |
| `provider_count` | cbbd_line_consensus | Number of providers that had a line for this game |
| `home_elo_start` / `away_elo_start` | bronze/cbbd/games | Elo rating entering the game (pre-game expectation) |
| `excitement` | bronze/cbbd/games | CBBD excitement index — higher = closer/more exciting game |
| `pace` | cbbd_game_fact | Estimated possessions per 40 minutes |
| `avg_margin` | team_season_features | Average point differential (pts_for − pts_against) per game |
| `sos` | team_season_features | Strength of schedule (mean ELO of opponents faced) |
| `fg_pct` / `fg3_pct` / `ft_pct` | team_season_features | Shooting efficiency: field goal %, 3-point %, free throw % |
| `opp_fg_pct` | team_season_features | Opponent field goal % (defensive quality) |
| `avg_reb_margin` | team_season_features | Average rebounding margin per game |
| `ast_to_ratio` | team_season_features | Assist-to-turnover ratio |
| `avg_stl_blk` | team_season_features | Average steals + blocks per game |
| `avg_possessions` | team_season_features | Tempo proxy (FGA − OREB + TO + 0.475×FTA) |
| `heat_delta` | heat_scores | actual_margin − expected_margin for a single game |
| `heat_1g` / `heat_3g` / `heat_5g` | heat_scores | Rolling heat averages over 1, 3, 5 games |
| `diff_*` | pairwise_features | team_low stat minus team_high stat (positive = team_low advantage) |
| `seed_product` | pairwise_features | seed_low × seed_high (interaction term) |
| `consensus_low_spread_filled` | pairwise_features | Betting spread for team_low (0.0 when unavailable) |
| `label` | pairwise_features | 1 if team_low won, 0 if team_high won (only present in training rows) |
| `Pred` | submission | P(team_low beats team_high), dynamically clipped by seed matchup |